# 03 · Model Training
Walks through the full XGBoost training pipeline for all three risk models.

**Sections:**
1. Hyperparameter overview
2. Cross-validation (stratified 5-fold)
3. Final model training and saving
4. Cross-model comparison

> **Note:** This notebook loads *pre-trained* models for inspection.  
> To retrain, run the training scripts directly:
> ```bash
> venv/bin/python src/ml/training/train_diabetes.py
> venv/bin/python src/ml/training/train_hypertension.py
> venv/bin/python src/ml/training/train_cvd.py
> ```

In [ ]:
import sys, warnings, json
from pathlib import Path
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score, accuracy_score

sns.set_theme(style='darkgrid', palette='muted')
MODELS_DIR = Path('../models/saved')
print('Ready.')

## 1 · Hyperparameter configuration

In [ ]:
from config.settings import DIABETES_CONFIG, HYPERTENSION_CONFIG

for name, cfg in [('Diabetes', DIABETES_CONFIG), ('Hypertension', HYPERTENSION_CONFIG)]:
    params = cfg['model_params']
    targets = cfg['validation']
    print(f'\n── {name} ──')
    for k, v in params.items():
        print(f'  {k:30} {v}')
    print(f'  Target accuracy: {targets["target_accuracy"]:.0%}')
    print(f'  Target AUC:      {targets["target_auc"]}')

## 2 · Load pre-trained diabetes model and metadata

In [ ]:
import joblib

def load_latest(prefix):
    files = sorted(MODELS_DIR.glob(f'{prefix}_model_*.joblib'))
    if not files:
        return None, None
    model_path = files[-1]
    meta_path  = model_path.with_suffix('').with_suffix('').parent / (model_path.stem + '_metadata.json')
    model = joblib.load(model_path)
    meta  = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    return model, meta

diab_model, diab_meta = load_latest('diabetes')
htn_model,  htn_meta  = load_latest('hypertension')

for name, meta in [('Diabetes', diab_meta), ('Hypertension', htn_meta)]:
    if meta:
        print(f'{name}: trained {meta.get("trained_at","?")[:10]}  |  '
              f'{meta.get("n_samples",0):,} samples  |  '
              f'{meta.get("n_features",0)} features  |  '
              f'best_iteration={meta.get("best_iteration","?")}')
    else:
        print(f'{name}: not found — run training script')

## 3 · 5-fold cross-validation (diabetes model demo)

In [ ]:
from src.ml.data.diabetes_preprocessor import DiabetesPreprocessor

DATA_DIR = '../data/raw/2017_2018'
prep = DiabetesPreprocessor()
X, y = prep.load_and_preprocess(DATA_DIR)

xgb_params = {
    k: v for k, v in DIABETES_CONFIG['model_params'].items()
    if k != 'early_stopping_rounds'
}

model = XGBClassifier(**xgb_params, eval_metric='logloss', verbosity=0)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(
    model, X, y,
    cv=cv,
    scoring=['accuracy', 'roc_auc', 'f1'],
    return_train_score=True,
    n_jobs=-1,
)

print('5-fold CV results:')
for metric in ('accuracy', 'roc_auc', 'f1'):
    test_vals  = scores[f'test_{metric}']
    train_vals = scores[f'train_{metric}']
    print(f'  {metric:10}  CV={test_vals.mean():.4f} ± {test_vals.std():.4f}  '
          f'Train={train_vals.mean():.4f} ± {train_vals.std():.4f}')

In [ ]:
# Box plot of fold-level AUC scores
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(
    [scores['test_accuracy'], scores['test_roc_auc'], scores['test_f1']],
    labels=['Accuracy', 'ROC-AUC', 'F1'],
    patch_artist=True,
    boxprops=dict(facecolor='steelblue', alpha=0.6),
)
ax.set_title('Diabetes model — 5-fold CV metric distribution')
ax.set_ylim(0.5, 1.05)
ax.axhline(0.95, ls='--', color='tomato', lw=1, label='0.95 target')
ax.legend()
plt.tight_layout()
plt.show()

## 4 · Comparing cross-validation vs final model metrics

In [ ]:
def load_eval(prefix):
    files = sorted(MODELS_DIR.glob(f'{prefix}_evaluation_*.json'))
    if not files:
        return None
    return json.loads(files[-1].read_text())

diab_eval = load_eval('diabetes')
htn_eval  = load_eval('hypertension')

rows = []
for name, ev in [('Diabetes', diab_eval), ('Hypertension', htn_eval)]:
    if ev:
        bm = ev['evaluation']['basic_metrics']
        rows.append({
            'Model':     name,
            'Accuracy':  bm['accuracy'],
            'ROC-AUC':   bm['roc_auc'],
            'Recall':    bm['recall'],
            'Precision': bm['precision'],
            'F1':        bm['f1_score'],
            'Brier':     bm['brier_score'],
        })

summary = pd.DataFrame(rows).set_index('Model')
summary.style.format('{:.4f}').background_gradient(cmap='Blues', axis=None)

## 5 · Training insights

- **Early stopping** (`early_stopping_rounds=20`) prevents overfitting; diabetes model converged
  at ~28 trees out of 200 maximum.
- The **class imbalance** (≈15% positive) naturally inflates accuracy — AUC and F1 are more
  meaningful metrics.
- The **hypertension model** achieves lower AUC than diabetes because BP readings are
  intentionally excluded from features (they define the target).

➡️ Continue to `04_model_evaluation.ipynb`